# Experiment 1 — Báo cáo kết quả (profile `paper`)

Notebook này đọc kết quả đã chạy sẵn trong `outputs/` và tạo:

1. **Một bảng kết quả cuối cùng** (mỗi optimizer một dòng, gộp trung bình ± độ lệch chuẩn trên các seed).
2. **Một biểu đồ đường hội tụ**: trục đứng là sai số dự đoán (MSE / RMSE), trục ngang là
   **ngân sách lan truyền ngược** (`backprop_calls`), vẽ đồng thời **train** (nét đứt) và **val** (nét liền)
   để xem đường đã *đi ngang* hay chưa.
3. **Bảng chẩn đoán plateau** — định lượng mức độ "đi ngang" của đoạn cuối mỗi đường.

> Giả định: đã có `comparison.csv`, `comparison_agg.csv` và `seed*/metrics.csv` trong thư mục kết quả
> của một tỉ lệ dữ liệu (ví dụ `outputs/experiment1/data100/`).

## 0. Cấu hình

Chỉ cần chỉnh ô dưới. Để `RESULTS_DIR = None` sẽ tự dò thư mục `data*` có nhiều seed nhất.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ #
# Thư mục kết quả cho MỘT tỉ lệ dữ liệu.
#   None  -> tự dò thư mục data* nhiều seed nhất.
#   Sau khi chạy src/run_experiment1.py, đặt thẳng, ví dụ:
#   RESULTS_DIR = 'outputs/data4'
# Bố cục mới (KHÔNG còn mức 'experiment1'): mỗi lần chạy = một thư mục con riêng
#   outputs/<dataTAG>/<YYYYmmdd_HHMMSS_fff>__seed<seed>/metrics.csv
# Kết quả cũ (trước 2026-09) nằm ở outputs_old/.
RESULTS_DIR = 'outputs/data4'

METRIC = 'rmse'            # 'rmse' hoặc 'mse'  (xem phần 1 để biết nên dùng cái nào)
LOG_Y  = True             # trục đứng thang log10 (nên bật khi các đường cách nhau nhiều bậc)
PLATEAU_LAST_FRAC = 0.30  # xét 30% cuối của ngân sách để kết luận 'đã đi ngang chưa'
PLATEAU_REL_TOL   = 0.05  # |thay đổi tương đối| < 5% trên đoạn cuối => coi như hội tụ ngang
OPT_ORDER  = ['AdamW', 'MCSDCA-odLD', 'MCSDCA-udLD']
OPT_COLORS = {'AdamW': '#1f77b4', 'MCSDCA-odLD': '#d62728', 'MCSDCA-udLD': '#2ca02c'}
# ------------------------------------------------------------------ #

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

def _seed_of(name):
    m = re.search(r'seed(\d+)', name)
    return int(m.group(1)) if m else -1

def _run_dirs(d):
    """Thư mục-con chứa metrics.csv (bố cục cũ 'seed*/' + 'sanity/' lẫn mới '*__seedN/')."""
    return [p.parent for p in d.glob('*/metrics.csv') if p.parent.name != 'sanity']

def _latest_per_seed(run_dirs):
    """Một seed chạy lại nhiều lần -> giữ thư mục mới nhất (theo mtime của metrics.csv)."""
    best = {}
    for p in run_dirs:
        s = _seed_of(p.name)
        mt = (p / 'metrics.csv').stat().st_mtime
        if s not in best or mt > best[s][0]:
            best[s] = (mt, p)
    return [best[s][1] for s in sorted(best)]

def _search_bases(root):
    # mới: outputs/data*  ;  cũ: outputs_old/**/experiment1/data*
    yield root / 'outputs'
    for legacy in ('experiment1', 'outputs/experiment1', 'hus/experiment1', 'hus/outputs/experiment1'):
        yield root / 'outputs_old' / legacy

def _find_results_dirs(root):
    found = []
    for base in _search_bases(root):
        if not base.exists():
            continue
        for d in sorted(base.glob('data*')):
            runs = _latest_per_seed(_run_dirs(d))
            comp = d / 'comparison.csv'
            if not (comp.exists() and runs):
                continue
            try:
                frac = float(pd.read_csv(comp)['data_fraction'].iloc[0])
            except Exception:
                frac = 0.0
            found.append((len(runs), frac, d))
    return found

if RESULTS_DIR is None:
    cands = _find_results_dirs(REPO_ROOT)
    if not cands:
        raise FileNotFoundError(
            'Không thấy thư mục kết quả nào có comparison.csv + */metrics.csv. Đặt RESULTS_DIR thủ công.')
    RESULTS_DIR = max(cands, key=lambda t: (t[0], t[1]))[2]   # nhiều seed nhất, rồi frac lớn nhất
    print('Tự dò được:', *[f'{d.relative_to(REPO_ROOT)}({n} seed, frac={f:g})' for n, f, d in cands])
else:
    RESULTS_DIR = Path(RESULTS_DIR)
    if not RESULTS_DIR.is_absolute():
        RESULTS_DIR = (REPO_ROOT / RESULTS_DIR).resolve()

assert RESULTS_DIR.exists(), f'Không tồn tại: {RESULTS_DIR}'
METRIC = METRIC.lower()
assert METRIC in ('rmse', 'mse')
METRIC_LABEL = METRIC.upper()

def to_metric(x):
    x = np.asarray(x, dtype=float)
    return np.sqrt(x) if METRIC == 'rmse' else x

print('REPO_ROOT   =', REPO_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('run dirs    =', [p.name for p in _latest_per_seed(_run_dirs(RESULTS_DIR))])
print('METRIC      =', METRIC_LABEL, '| LOG_Y =', LOG_Y)

## 1. Nên dùng MSE hay RMSE?

| | MSE | RMSE = √MSE |
|---|---|---|
| Thứ hạng optimizer | — | **y hệt** (biến đổi đơn điệu ⇒ kết luận không đổi) |
| Ý nghĩa | Chính là đại lượng **đang được tối ưu**: loss huấn luyện = `MSE + λ·SIGReg` | Sai số *điển hình*, **cùng đơn vị** với latent đích (norm ≈ 2–14 ở đây) |
| Phát hiện plateau | Phạt outlier nặng ⇒ độ dốc cuối đường **nhạy hơn** | Mượt hơn, ít bị vài batch xấu kéo lệch |
| Khi giá trị trải rộng | val_mse ở đây trải ~0.03 → ~1.0 (hơn 1 bậc) ⇒ vẽ tuyến tính **bẹp** đường tốt | Nén biên độ ⇒ đọc rõ hơn |

**Khuyến nghị**

- **Bảng cuối**: báo cáo *cả hai* — MSE để đối chiếu trực tiếp với loss huấn luyện, RMSE để diễn giải sai số.
- **Biểu đồ hội tụ**: dùng **RMSE trục tuyến tính** *hoặc* **MSE trục log**. Notebook mặc định
  `METRIC='rmse'` + `LOG_Y=True`; đổi `METRIC='mse'` nếu muốn nhìn thẳng giá trị hàm mục tiêu.
- Để đánh giá "đã hội tụ chưa", cứ nhìn **cùng một chỉ số cho cả train và val** — quan trọng là *độ dốc
  đoạn cuối*, không phải giá trị tuyệt đối.

## 2. Bảng kết quả cuối cùng

Gộp từ `comparison.csv` (mỗi seed một dòng) → trung bình ± độ lệch chuẩn theo optimizer.
Sắp xếp theo `val` tăng dần (tốt nhất lên đầu).

In [2]:
comp = pd.read_csv(RESULTS_DIR / 'comparison.csv')
ok = comp[comp['status'] == 'ok'].copy()
if len(ok) < len(comp):
    print('CẢNH BÁO: có', len(comp) - len(ok), 'lần chạy status != ok (bị loại khỏi bảng).')

MSE_BASES = ['train_mse', 'val_mse', 'rollout_mse_1', 'rollout_mse_3', 'rollout_mse_5']
recs = []
for opt in [o for o in OPT_ORDER if o in set(ok['optimizer'])]:
    g = ok[ok['optimizer'] == opt]
    rec = {'optimizer': opt,
           'n_seeds': int(g['seed'].nunique()),
           'data_fraction': float(g['data_fraction'].iloc[0]),
           'backprop_calls': int(g['backprop_calls'].max())}
    for base in MSE_BASES:
        v = to_metric(g[base].to_numpy())
        stem = base.replace('_mse', '') or base   # train_mse->train, rollout_mse_1->rollout_1
        rec[f'{stem}_{METRIC}_mean'] = float(np.mean(v))
        rec[f'{stem}_{METRIC}_std'] = float(np.std(v, ddof=0))
    rec['train_val_gap_mean'] = float(g['train_val_gap'].mean())
    recs.append(rec)

final_tbl = (pd.DataFrame(recs)
             .sort_values(f'val_{METRIC}_mean')
             .reset_index(drop=True))

out_csv = RESULTS_DIR / 'report_final_table.csv'
final_tbl.to_csv(out_csv, index=False)
print('Đã lưu:', out_csv)
final_tbl

Đã lưu: D:\DCA\MCSDCA\outputs\hus\experiment1\data4\report_final_table.csv


,optimizer,n_seeds,data_fraction,backprop_calls,train_rmse_mean,train_rmse_std,val_rmse_mean,val_rmse_std,rollout_1_rmse_mean,rollout_1_rmse_std,rollout_3_rmse_mean,rollout_3_rmse_std,rollout_5_rmse_mean,rollout_5_rmse_std,train_val_gap_mean
0,AdamW,1,0.04,40000,0.093298,0.0,0.116016,0.0,0.119821,0.0,0.167395,0.0,0.208247,0.0,0.004755
1,MCSDCA-udLD,1,0.04,40011,0.594247,0.0,0.555654,0.0,0.570920,0.0,0.583487,0.0,0.592036,0.0,-0.044378
2,MCSDCA-odLD,1,0.04,40011,0.761893,0.0,0.767490,0.0,0.774125,0.0,0.831704,0.0,0.849916,0.0,0.008559


### 2b. Bảng gọn để đưa vào báo cáo (`mean ± std`)

In [3]:
def pm(mean, std, nd=5):
    return f'{mean:.{nd}f} ± {std:.{nd}f}'

pretty = pd.DataFrame({
    'optimizer': final_tbl['optimizer'],
    'seeds': final_tbl['n_seeds'],
    'backprop_calls': final_tbl['backprop_calls'],
    f'train {METRIC_LABEL}': [pm(m, s) for m, s in zip(final_tbl[f'train_{METRIC}_mean'], final_tbl[f'train_{METRIC}_std'])],
    f'val {METRIC_LABEL}':   [pm(m, s) for m, s in zip(final_tbl[f'val_{METRIC}_mean'],   final_tbl[f'val_{METRIC}_std'])],
    'train-val gap': [f'{v:.2e}' for v in final_tbl['train_val_gap_mean']],
    f'rollout-5 {METRIC_LABEL}': [pm(m, s) for m, s in zip(final_tbl[f'rollout_5_{METRIC}_mean'], final_tbl[f'rollout_5_{METRIC}_std'])],
})
# cột tham chiếu: cả MSE lẫn RMSE của val để tiện đối chiếu
val_mse_mean = ok.groupby('optimizer')['val_mse'].mean().reindex(final_tbl['optimizer']).to_numpy()
pretty['val_MSE (ref)'] = [f'{v:.5f}' for v in val_mse_mean]
pretty['val_RMSE (ref)'] = [f'{np.sqrt(v):.5f}' for v in val_mse_mean]
pretty

,optimizer,seeds,backprop_calls,train RMSE,val RMSE,train-val gap,rollout-5 RMSE,val_MSE (ref),val_RMSE (ref)
0,AdamW,1,40000,0.09330 ± 0.00000,0.11602 ± 0.00000,4.76e-03,0.20825 ± 0.00000,0.01346,0.11602
1,MCSDCA-udLD,1,40011,0.59425 ± 0.00000,0.55565 ± 0.00000,-4.44e-02,0.59204 ± 0.00000,0.30875,0.55565
2,MCSDCA-odLD,1,40011,0.76189 ± 0.00000,0.76749 ± 0.00000,8.56e-03,0.84992 ± 0.00000,0.58904,0.76749


In [4]:
# Bảng gốc do pipeline sinh ra (tham chiếu)
agg_path = RESULTS_DIR / 'comparison_agg.csv'
pd.read_csv(agg_path) if agg_path.exists() else 'không có comparison_agg.csv'

,optimizer,data_fraction,n_seeds_ok,val_mse_mean,val_mse_std,train_mse_mean,train_mse_std,train_val_gap_mean,train_val_gap_std,rollout_mse_1_mean,rollout_mse_1_std,rollout_mse_3_mean,rollout_mse_3_std,rollout_mse_5_mean,rollout_mse_5_std
0,AdamW,0.04,1,0.013460,0.0,0.008704,0.0,0.004755,0.0,0.014357,0.0,0.028021,0.0,0.043367,0.0
1,MCSDCA-odLD,0.04,1,0.589040,0.0,0.580481,0.0,0.008559,0.0,0.599269,0.0,0.691732,0.0,0.722358,0.0
2,MCSDCA-udLD,0.04,1,0.308751,0.0,0.353129,0.0,-0.044378,0.0,0.325950,0.0,0.340457,0.0,0.350507,0.0


## 3. Biểu đồ đường hội tụ theo ngân sách backprop

- **Trục ngang**: `backprop_calls` — số lần lan truyền ngược đã tiêu.
- **Trục đứng**: sai số dự đoán 1 bước theo `METRIC` (`rmse`/`mse`).
- **Nét liền = val**, **nét đứt = train**; dải mờ = ±1 std trên các seed.
- Đường *đi ngang* ở đoạn cuối ⇒ đã hết cải thiện trong ngân sách này.

In [ ]:
def load_history(results_dir):
    frames = []
    for rd in _latest_per_seed(_run_dirs(results_dir)):
        df = pd.read_csv(rd / 'metrics.csv')
        df['seed'] = _seed_of(rd.name)
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f'Không có */metrics.csv trong {results_dir}')
    h = pd.concat(frames, ignore_index=True)
    return h[h.get('status', 'ok').fillna('ok') == 'ok'].copy()

hist = load_history(RESULTS_DIR)
n_seeds = hist['seed'].nunique()
print('seeds:', sorted(hist['seed'].unique()), '| optimizers:', sorted(hist['optimizer'].unique()))

def agg_curve(hist, opt, ycol, n_grid=80):
    """Nội suy đường của từng seed lên lưới chung -> (grid, mean, std, n_seed).

    Giá trị trả về là RAW (chưa qua to_metric). Đường sai số: bọc to_metric ở nơi gọi.
    """
    if ycol not in hist.columns:
        return None
    sub = hist[(hist['optimizer'] == opt)][['seed', 'backprop_calls', ycol]].dropna()
    if sub.empty:
        return None
    per_seed = sub.groupby('seed')['backprop_calls']
    lo = float(per_seed.min().max())   # điểm bắt đầu chung
    hi = float(per_seed.max().min())   # điểm kết thúc chung
    if hi <= lo:                       # các seed không chồng lấn -> gộp thô
        lo, hi = float(sub['backprop_calls'].min()), float(sub['backprop_calls'].max())
    grid = np.linspace(lo, hi, n_grid)
    stacks = []
    for _, g in sub.groupby('seed'):
        g = g.sort_values('backprop_calls')
        stacks.append(np.interp(grid, g['backprop_calls'], g[ycol]))
    Y = np.vstack(stacks)
    return grid, Y.mean(0), Y.std(0), Y.shape[0]

fig, ax = plt.subplots(figsize=(9.5, 5.8))
present = [o for o in OPT_ORDER if o in set(hist['optimizer'])]
for opt in present:
    for split, ycol, ls in [('train', 'train_mse', '--'), ('val', 'val_mse', '-')]:
        res = agg_curve(hist, opt, ycol)
        if res is None:
            continue
        x, mean, std, ns = res
        mean_m = to_metric(mean)
        ax.plot(x, mean_m, ls=ls, lw=2, color=OPT_COLORS.get(opt), label=f'{opt} · {split}')
        if ns > 1:
            band = to_metric(np.clip(mean + std, 0, None)) - mean_m
            ax.fill_between(x, mean_m - band, mean_m + band, color=OPT_COLORS.get(opt), alpha=0.15, lw=0)

if LOG_Y:
    ax.set_yscale('log')
ax.set_xlabel('Ngân sách lan truyền ngược  (backprop_calls)')
ax.set_ylabel(f'{METRIC_LABEL} dự đoán 1 bước')
ax.set_title(f'Đường hội tụ theo ngân sách backprop — {RESULTS_DIR.name} '
             f'({METRIC_LABEL}, {n_seeds} seed)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(ncol=3, fontsize=8, loc='upper right')
fig.tight_layout()
fig_path = RESULTS_DIR / 'report_convergence.png'
fig.savefig(fig_path, dpi=150)
print('Đã lưu:', fig_path)
plt.show()

### 3b. (Bổ sung) Đường hội tụ của rollout-5

Sai số rollout 5 bước trên val — chỉ số dùng để xếp hạng chính trong kế hoạch thí nghiệm.

In [ ]:
if 'rollout_mse_5' in hist.columns and hist['rollout_mse_5'].notna().any():
    fig2, ax2 = plt.subplots(figsize=(9.5, 5.0))
    for opt in present:
        res = agg_curve(hist, opt, 'rollout_mse_5')
        if res is None:
            continue
        x, mean, std, ns = res
        mean_m = to_metric(mean)
        ax2.plot(x, mean_m, lw=2, color=OPT_COLORS.get(opt), label=f'{opt} · rollout-5 (val)')
        if ns > 1:
            band = to_metric(np.clip(mean + std, 0, None)) - mean_m
            ax2.fill_between(x, mean_m - band, mean_m + band, color=OPT_COLORS.get(opt), alpha=0.15, lw=0)
    if LOG_Y:
        ax2.set_yscale('log')
    ax2.set_xlabel('Ngân sách lan truyền ngược  (backprop_calls)')
    ax2.set_ylabel(f'{METRIC_LABEL} rollout-5 (val)')
    ax2.set_title(f'Hội tụ rollout-5 — {RESULTS_DIR.name} ({METRIC_LABEL}, {n_seeds} seed)')
    ax2.grid(True, which='both', alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(RESULTS_DIR / 'report_convergence_rollout5.png', dpi=150)
    plt.show()
else:
    print('Không có cột rollout_mse_5.')

## 3c. Chẩn đoán sụp đổ biểu diễn — encoder hay predictor?

Mục tiêu LeWM **không** stop-gradient trên target, nên SIGReg là lực chống sụp đổ duy nhất.
Các cột `col_*` (one-step, trên val) tách vị trí sụp đổ:

| Cột | Ý nghĩa | Sụp khi |
|---|---|---|
| `col_enc_emb_var_mean` | phương sai trung bình đầu ra encoder | → 0 |
| `col_enc_dead_dim_frac` | tỉ lệ chiều encoder có var < 1e-4 | → 1 |
| `col_enc_emb_norm_mean` | ‖emb‖ trung bình | → nhỏ |
| `col_pred_target_var_ratio` | var(pred) / var(target), cắt trần ở 10 | ≪ 1 ⇒ **predictor sụp** |
| `col_pred_target_norm_ratio` | ‖pred‖ / ‖target‖, cắt trần ở 10 | ≪ 1 |

**Đọc:** `enc_*` thấp + `dead_dim_frac` cao ⇒ **encoder sụp**. `enc_*` khỏe nhưng
`pred_target_var_ratio` ≪ 1 ⇒ **predictor sụp**. Cả hai thấp ⇒ encoder kéo theo.

In [ ]:
COLLAPSE_PANELS = [
    ('col_enc_emb_var_mean',       'var(emb) encoder',           True),
    ('col_enc_dead_dim_frac',      'tỉ lệ chiều encoder "chết"',  False),
    ('col_pred_target_var_ratio',  'var(pred) / var(target)',     False),
    ('col_pred_target_norm_ratio', '‖pred‖ / ‖target‖',           False),
]
avail = [p for p in COLLAPSE_PANELS if p[0] in hist.columns]
if not avail:
    print('metrics.csv chưa có cột col_* — chạy lại bằng src/run_experiment1.py bản mới.')
else:
    fig, axes = plt.subplots(1, len(avail), figsize=(4.6 * len(avail), 4.2), squeeze=False)
    for ax, (col, title, logy) in zip(axes[0], avail):
        for opt in present:
            res = agg_curve(hist, opt, col)
            if res is None:
                continue
            x, mean, std, ns = res
            ax.plot(x, mean, lw=2, color=OPT_COLORS.get(opt), label=opt)
            if ns > 1:
                ax.fill_between(x, mean - std, mean + std, color=OPT_COLORS.get(opt), alpha=0.15, lw=0)
        if 'ratio' in col:
            ax.axhline(1.0, color='0.4', ls=':', lw=1)   # mốc lành mạnh
        if logy:
            ax.set_yscale('log')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('backprop_calls')
        ax.grid(True, which='both', alpha=0.3)
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f'Chẩn đoán sụp đổ biểu diễn — {RESULTS_DIR.name} ({n_seeds} seed)', y=1.02)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'report_collapse.png', dpi=150, bbox_inches='tight')
    print('Đã lưu:', RESULTS_DIR / 'report_collapse.png')
    plt.show()

    # bảng giá trị cuối
    last = (hist.sort_values('backprop_calls').groupby(['optimizer', 'seed']).tail(1))
    cols = [c for c, _, _ in avail] + ['col_pred_var_mean', 'col_target_var_mean', 'col_enc_emb_norm_mean']
    cols = [c for c in cols if c in last.columns]
    display(last.groupby('optimizer')[cols].mean().reindex([o for o in OPT_ORDER if o in set(last['optimizer'])]))

## 4. Chẩn đoán plateau — "đường đã đi ngang chưa?"

Với mỗi optimizer × {train, val}, xét đoạn cuối chiếm `PLATEAU_LAST_FRAC` bề rộng ngân sách và tính:

- **rel_drop_tail**: `(y_đầu_đoạn − y_cuối_đoạn) / y_đầu_đoạn` — dương = *vẫn còn giảm*.
- **rel_span_tail**: `(max − min) / mean` trên đoạn cuối — lớn = *còn dao động mạnh*.
- **slope_per_1k**: độ dốc hồi quy tuyến tính (đơn vị `METRIC` trên mỗi 1000 backprop).
- **verdict**:
  - `đã đi ngang` nếu `|rel_drop_tail| < PLATEAU_REL_TOL` **và** `rel_span_tail < 2·PLATEAU_REL_TOL`;
  - `chưa hội tụ (còn giảm)` nếu `rel_drop_tail ≥ PLATEAU_REL_TOL`;
  - `còn dao động` nếu không giảm rõ nhưng biên độ đoạn cuối lớn.

In [ ]:
def plateau_row(opt, split, ycol):
    res = agg_curve(hist, opt, ycol)
    if res is None:
        return None
    x, mean, _, _ = res
    y = to_metric(mean)
    span = x.max() - x.min()
    mask = x >= (x.max() - PLATEAU_LAST_FRAC * span) if span > 0 else np.ones_like(x, bool)
    xt, yt = x[mask], y[mask]
    y0, y1 = float(yt[0]), float(yt[-1])
    rel_drop = (y0 - y1) / abs(y0) if y0 else np.nan
    rel_span = (yt.max() - yt.min()) / abs(yt.mean()) if yt.mean() else np.nan
    slope = np.polyfit(xt, yt, 1)[0] if len(xt) > 1 else 0.0
    if abs(rel_drop) < PLATEAU_REL_TOL and rel_span < 2 * PLATEAU_REL_TOL:
        verdict = 'đã đi ngang'
    elif rel_drop >= PLATEAU_REL_TOL:
        verdict = 'chưa hội tụ (còn giảm)'
    else:
        verdict = 'còn dao động'
    return {'optimizer': opt, 'split': split,
            f'{METRIC_LABEL}_cuối': round(y1, 6),
            'rel_drop_tail': round(float(rel_drop), 4),
            'rel_span_tail': round(float(rel_span), 4),
            'slope_per_1k': f'{slope * 1000:.2e}',
            'verdict': verdict}

rows = []
for opt in present:
    for split, ycol in [('train', 'train_mse'), ('val', 'val_mse')]:
        r = plateau_row(opt, split, ycol)
        if r:
            rows.append(r)
plateau_tbl = pd.DataFrame(rows)
plateau_tbl.to_csv(RESULTS_DIR / 'report_plateau.csv', index=False)
print(f'Đoạn cuối = {PLATEAU_LAST_FRAC:.0%} bề rộng ngân sách; ngưỡng rel = {PLATEAU_REL_TOL:.0%}')
plateau_tbl

## 5. Nhận xét nhanh

- Nếu **val** đi ngang mà **train** vẫn giảm ⇒ bắt đầu overfit / hết lợi ích từ thêm ngân sách.
- Nếu **cả hai** còn dốc ⇒ tăng `--paper-epochs` (hoặc `--eval-budget`) rồi chạy lại.
- Nếu **cả hai** đã ngang và `rel_span_tail` nhỏ ⇒ ngân sách hiện tại đủ để so sánh công bằng.
- So sánh chéo optimizer nên đọc ở **cùng `backprop_calls`** (trục ngang), không phải cùng số epoch.

Các file đã ghi vào thư mục kết quả: `report_final_table.csv`, `report_plateau.csv`,
`report_convergence.png`, `report_convergence_rollout5.png`.